## Initial setup

This will setup a connection and make an API client call. After a client makes an API call, it comes back with messages in the form of a LIST. Where is the role/content pairs. There is no memory in this exanple

In [ ]:
import anthropic
from dotenv import load_dotenv

# Load environment keys, the anthropic one
load_dotenv()

client = anthropic.Anthropic() # Actual client for anthropic

# An API call
message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024, # Max length of response
    messages=[
        {
            "role": "user",
            "content": "Hey, are you a scam? Reply suspiciously."
        }
    ]
)

print(message.content[0].text)

The response is a list of blocks with a `type` and `text`. 

## Part 2: Your First Tool

**Tools are functions we describe to the model in JSON.**

The model doesn't run the tools, I do, but the model will call them. This is the heart of every agentic system ever built.

In [2]:
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()

# Step 1: Define my tool
tools = [
    {
        "name": "classify_message",
        "description": "Classifies whether a message is a scam or legitimate.",
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to classify"
                },
                "is_scam": {
                    "type": "boolean",
                    "description": "True if the message is a scam. False if legitimate"
                },
                "confidence": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "Confidence level of the classification"
                },
                "reason": {
                    "type": "string",
                    "description": "Brief reason for the classification"
                }
            },
            "required": ["message", "is_scam", "confidence", "reason"]
        }

    }
]

# Step 2: A fake scam message to test with
test_message = "Hey friend! I am prince of Nigeria and I need your help. Send me $500 and I will send you $50,000 back. Very urgent!!!"

# Step 3: Send to Claude with the tool available
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=tools,
    messages=[
        {
            "role": "user",
            "content": f"Classify this message: {test_message}"
        }
    ]
)

# Step 4: Inspect the response
print("Stop reason: ", response.stop_reason) # Why the model stopped performing
print("Content blocks: ", response.content) # All responses(from the list of blocks)

# Step 5: Check if CLaude wants to use the tool
for block in response.content:
    if block.type == "tool_use":
        print("\n -- Tool Call Detected -- ")
        print("Tool name: ", block.name)
        print("Arguments: ", block.input)

Stop reason:  tool_use
Content blocks:  [ToolUseBlock(id='toolu_01LgYEHVDHoHvEJgwzguFaMr', caller=DirectCaller(type='direct'), input={'message': 'Hey friend! I am prince of Nigeria and I need your help. Send me $500 and I will send you $50,000 back. Very urgent!!!', 'is_scam': True, 'confidence': 'high', 'reason': 'Classic Nigerian prince advance fee scam with unrealistic return promise, urgency tactics, and request for upfront payment'}, name='classify_message', type='tool_use')]

 -- Tool Call Detected -- 
Tool name:  classify_message
Arguments:  {'message': 'Hey friend! I am prince of Nigeria and I need your help. Send me $500 and I will send you $50,000 back. Very urgent!!!', 'is_scam': True, 'confidence': 'high', 'reason': 'Classic Nigerian prince advance fee scam with unrealistic return promise, urgency tactics, and request for upfront payment'}


What's important here is the following:

* You described a function
* The model decided to call the it on its own
* **The tool hasn't run yet. Claude said what it wants to call(`classify_message`) and with what arguments.**

Result:

* The model stopped and said "I need to call a tool"
* Identified a scam with `high` confidence, and gave a solid reason. **No classification logic used**. Instead, the model did it from my tool description.

## Phase 3: Closing the loop

* Receive the tool call from Claude(like before)
* Actually "run" the tool, which is a new python function
* Send the result back to Claude
* Get a final human-readable response

In [3]:
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()

# Step 1: Define my tool
tools = [
    {
        "name": "classify_message",
        "description": "Classifies whether a message is a scam or legitimate.",
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to classify"
                },
                "is_scam": {
                    "type": "boolean",
                    "description": "True if the message is a scam. False if legitimate"
                },
                "confidence": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "Confidence level of the classification"
                },
                "reason": {
                    "type": "string",
                    "description": "Brief reason for the classification"
                }
            },
            "required": ["message", "is_scam", "confidence", "reason"]
        }

    }
]

## Tool
def classify_message(message, is_scam, confidence, reason):
    label = "SCAM" if is_scam else "LEGITIMATE"
    return f"Classification: {label} | Confidence: {confidence} | Reason: {reason}"

test_message = "Hey friend! I am prince of Nigeria and I need your help. Send me $500 and I will send you $50,000 back. Very urgent!!!"

# Turn 1: User(Me) sends message to Claude
messages = [
    {"role": "user", "content": f"Classify this message and roast it if it's a scam: {test_message}"}
]

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=tools,
    messages=messages
)

# Step 2: Claude calls a tool, we run it and send the result back
if response.stop_reason == "tool_use":
    tool_block = next(b for b in response.content if b.type == "tool_use")

    # Run the function
    result = classify_message(**tool_block.input) # The input it wants to put into that function
    print("Tool result:", result)

    # Now build the next messages to send back. Add what CLaude said
    messages.append({"role": "assistant", "content": response.content})

    # Now add out response. This is now conversation history
    messages.append({
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_block.id,
                "content": result
            }
        ]
    })

# Turn 3: Claude sees the tool result and gives final response --

final_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=tools,
    messages=messages
)


print("\n--- Final Response ---")
print(final_response.content[0].text)

Tool result: Classification: SCAM | Confidence: high | Reason: Classic Nigerian prince advance fee scam with unrealistic return promise and urgency tactics

--- Final Response ---
🚨 **SCAM ALERT** 🚨

Oh wow, a Nigerian prince! How original! 🙄 Let me break down this masterpiece of deception:

**The Roast:**
- "Hey friend!" - Because nothing says "legitimate royalty" like starting with the greeting equivalent of a used car salesman
- Claims to be Nigerian royalty but apparently learned English from a spam bot having a stroke
- Offering a 10,000% return on investment - even the best hedge funds are jealous of these fictional math skills!
- "Very urgent!!!" - Yes, very urgent because his creativity expires at midnight and he needs to copy-paste this to 10,000 more people
- $500 for $50,000? Even monopoly money has better exchange rates

**Red flags so big they could be seen from space:**
- Unsolicited contact claiming royal status
- Advance fee fraud (pay money upfront)
- Unrealistic promi

Read the `messages` list after the tool call and understand what it looks like at that point. It's now 3 entries:

1. Your original user message
2. Claude's response (which contains the tool call)
3. Your tool result (sent back as a user turn with a special tool_result type)

That 3-entry structure is the fundamental pattern of all agentic systems. LangGraph, CrewAI, AutoGen — they all just automate building this list. Now you're doing it by hand, which means you actually understand it.

## Phase 4: Multiple messages at a time

* Handle multiple messages in a loop
* Adding a second tool: "Generate roast response"